# RSF Hyperparameter Tuning + Repeated Cross-Validated Model Comparison

Tunes Random Survival Forest via grid search (train split only, 5-fold CV), then compares Cox PH, tuned RSF, and Gradient Boosted Survival Analysis via repeated 10-seed x 5-fold cross-validation.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
BASE = '../../'
USYD = ['#E64626', '#DAA8A2', '#71A399', '#8F9EC9', '#1A345E']

df = pd.read_csv(BASE + 'modeling_dataset.csv')
print(f'{len(df):,} patients, split counts: {df.split.value_counts().to_dict()}')

3,050 patients, split counts: {'train': 2440, 'test': 610}


## 1. Covariates + train/test split 

In [2]:
COMORBID_COLS = ['cat_Renal', 'cat_FluidElectrolyte', 'cat_Metabolic', 'cat_Respiratory',
                  'cat_MentalHealthSubstance', 'cat_Musculoskeletal', 'cat_Hematologic', 'cat_Hepatic',
                  'cat_Neurologic', 'cat_Oncologic', 'cat_EndocrineOther', 'cat_ConstitutionalFrailty']
UTILIZATION_COLS = ['baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
                     'baseline_n_episodes', 'baseline_history_days']

surv_df = df[['years_followup', 'incident_cvd', 'age', 'split'] + COMORBID_COLS + UTILIZATION_COLS].copy()
surv_df['sex_M'] = (df['sex'] == 'M').astype(int)
surv_df['incident_cvd'] = surv_df['incident_cvd'].astype(int)

COVARIATES = ['age', 'sex_M'] + COMORBID_COLS + UTILIZATION_COLS

train = surv_df[surv_df['split'] == 'train'].reset_index(drop=True)
test = surv_df[surv_df['split'] == 'test'].reset_index(drop=True)
print(f'train: {len(train):,} ({train.incident_cvd.sum()} events), test: {len(test):,} ({test.incident_cvd.sum()} events)')

X_train, X_test = train[COVARIATES], test[COVARIATES]
y_train = Surv.from_arrays(event=train['incident_cvd'].astype(bool), time=train['years_followup'])
y_test = Surv.from_arrays(event=test['incident_cvd'].astype(bool), time=test['years_followup'])

train: 2,440 (378 events), test: 610 (94 events)


## 2. RSF hyperparameter grid search

5-fold CV on the train split only, scored by concordance index. 45 combinations x 5 folds.

In [3]:
param_grid = {
    'min_samples_leaf': [5, 10, 15, 20, 30],
    'max_features': ['sqrt', 'log2', 0.5],
    'n_estimators': [300, 500, 1000],
}

cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(cv_splitter.split(X_train, train['incident_cvd']))

rsf_base = RandomSurvivalForest(n_jobs=-1, random_state=RANDOM_STATE)
grid = GridSearchCV(rsf_base, param_grid, cv=cv_splits, n_jobs=1, refit=False)
grid.fit(X_train, y_train)

cv_results = pd.DataFrame(grid.cv_results_)[['param_min_samples_leaf', 'param_max_features', 'param_n_estimators',
                                              'mean_test_score', 'std_test_score']]
cv_results = cv_results.rename(columns={'mean_test_score': 'mean_cv_cindex', 'std_test_score': 'std_cv_cindex'})
cv_results = cv_results.sort_values('mean_cv_cindex', ascending=False).reset_index(drop=True)
cv_results.to_csv('rsf_grid_search_results.csv', index=False)

print(f'Best params (5-fold CV mean C-index = {grid.best_score_:.4f}): {grid.best_params_}')
print()
print('=== Top 10 combinations ===')
cv_results.head(10).round(4)

Best params (5-fold CV mean C-index = 0.7226): {'max_features': 'sqrt', 'min_samples_leaf': 10, 'n_estimators': 300}

=== Top 10 combinations ===


,param_min_samples_leaf,param_max_features,param_n_estimators,mean_cv_cindex,std_cv_cindex
0,10,sqrt,300,0.7226,0.0184
1,10,log2,300,0.7226,0.0184
2,10,sqrt,500,0.7225,0.0189
3,10,log2,500,0.7225,0.0189
4,10,sqrt,1000,0.7221,0.0202
5,10,log2,1000,0.7221,0.0202
6,15,log2,1000,0.7205,0.0181
7,15,sqrt,1000,0.7205,0.0181
8,15,log2,500,0.7205,0.0178
9,15,sqrt,500,0.7205,0.0178


## 3. Refit tuned RSF, evaluate on held-out test

In [4]:
best_params = grid.best_params_
rsf_tuned = RandomSurvivalForest(n_jobs=-1, random_state=RANDOM_STATE, **best_params)
rsf_tuned.fit(X_train, y_train)

rsf_tuned_pred = rsf_tuned.predict(X_test)
rsf_tuned_cindex = concordance_index_censored(y_test['event'], y_test['time'], rsf_tuned_pred)[0]

rsf_default = RandomSurvivalForest(n_estimators=500, min_samples_leaf=15, max_features='sqrt',
                                    n_jobs=-1, random_state=RANDOM_STATE)
rsf_default.fit(X_train, y_train)
rsf_default_pred = rsf_default.predict(X_test)
rsf_default_cindex = concordance_index_censored(y_test['event'], y_test['time'], rsf_default_pred)[0]

print(f'RSF, untuned defaults (02_random_survival_forest.ipynb):  test C-index = {rsf_default_cindex:.4f}')
print(f'RSF, tuned via grid search:                                test C-index = {rsf_tuned_cindex:.4f}')
print(f'Tuned hyperparameters: {best_params}')

RSF, untuned defaults (02_random_survival_forest.ipynb):  test C-index = 0.6774
RSF, tuned via grid search:                                test C-index = 0.6808
Tuned hyperparameters: {'max_features': 'sqrt', 'min_samples_leaf': 10, 'n_estimators': 300}


## 4. Repeated cross-validated comparison: Cox PH vs. tuned RSF vs. GBSA

10 seeds x 5-fold CV across the full cohort = 50 held-out C-index measurements per model.

In [5]:
SEEDS = list(range(10))
N_FOLDS = 5

X_all = surv_df[COVARIATES]
y_all = Surv.from_arrays(event=surv_df['incident_cvd'].astype(bool), time=surv_df['years_followup'])
event_all = surv_df['incident_cvd']

MODELS = {
    'Cox PH': lambda: None,  # fit via lifelines below (different API)
    'Random Survival Forest (tuned)': lambda: RandomSurvivalForest(n_jobs=-1, random_state=seed, **best_params),
    'Gradient Boosted Survival Analysis': lambda: GradientBoostingSurvivalAnalysis(random_state=seed),
}

rows = []
for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(X_all, event_all)):
        Xtr, Xte = X_all.iloc[tr_idx], X_all.iloc[te_idx]
        ytr, yte = y_all[tr_idx], y_all[te_idx]
        dtr = surv_df.iloc[tr_idx]
        dte = surv_df.iloc[te_idx]

        if yte['event'].sum() < 2:
            continue

        # Cox PH
        cph = CoxPHFitter(penalizer=0.001)
        cph.fit(dtr[['years_followup', 'incident_cvd'] + COVARIATES], duration_col='years_followup',
                event_col='incident_cvd', formula=' + '.join(COVARIATES))
        cox_risk = cph.predict_partial_hazard(dte[COVARIATES])
        cox_c = concordance_index_censored(yte['event'], yte['time'], cox_risk)[0]
        rows.append(dict(model='Cox PH', seed=seed, fold=fold_idx, c_index=cox_c))

        # RSF (tuned)
        rsf = RandomSurvivalForest(n_jobs=-1, random_state=seed, **best_params)
        rsf.fit(Xtr, ytr)
        rsf_c = concordance_index_censored(yte['event'], yte['time'], rsf.predict(Xte))[0]
        rows.append(dict(model='Random Survival Forest (tuned)', seed=seed, fold=fold_idx, c_index=rsf_c))

        # GBSA
        gbsa = GradientBoostingSurvivalAnalysis(random_state=seed)
        gbsa.fit(Xtr, ytr)
        gbsa_c = concordance_index_censored(yte['event'], yte['time'], gbsa.predict(Xte))[0]
        rows.append(dict(model='Gradient Boosted Survival Analysis', seed=seed, fold=fold_idx, c_index=gbsa_c))

    print(f'seed {seed}: done')

raw_df = pd.DataFrame(rows)
raw_df.to_csv('repeated_cv_survival_raw_per_fold.csv', index=False)
print(f'{len(raw_df)} total evaluations ({len(raw_df)//3} folds x 3 models)')

seed 0: done


seed 1: done


seed 2: done


seed 3: done


seed 4: done


seed 5: done


seed 6: done


seed 7: done


seed 8: done


seed 9: done
150 total evaluations (50 folds x 3 models)


## 5. Summary: C-index across 50 folds, and paired significance tests

In [6]:
summary = raw_df.groupby('model')['c_index'].agg(['mean', 'std', 'count']).round(4)
summary.columns = ['c_index_mean', 'c_index_std', 'n_folds']
summary = summary.sort_values('c_index_mean', ascending=False)
summary.to_csv('repeated_cv_survival_summary.csv')
print('=== Repeated CV C-index, 10 seeds x 5 folds = 50 evaluations per model ===')
print(summary)
print()

piv = raw_df.pivot_table(index=['seed', 'fold'], columns='model', values='c_index')

print('=== Paired Wilcoxon signed-rank tests (fold-matched, same splits) ===')
pairs = [('Random Survival Forest (tuned)', 'Cox PH'),
         ('Gradient Boosted Survival Analysis', 'Cox PH'),
         ('Random Survival Forest (tuned)', 'Gradient Boosted Survival Analysis')]
test_rows = []
for a, b in pairs:
    diffs = (piv[a] - piv[b]).dropna()
    stat, p = stats.wilcoxon(diffs)
    test_rows.append(dict(comparison=f'{a} vs {b}', median_diff=round(diffs.median(), 4),
                           n_pairs=len(diffs), wilcoxon_stat=stat, p_value=p))
    print(f'{a} vs {b}: median diff = {diffs.median():+.4f}, n={len(diffs)}, p = {p:.2e}')

pd.DataFrame(test_rows).to_csv('repeated_cv_survival_significance.csv', index=False)

=== Repeated CV C-index, 10 seeds x 5 folds = 50 evaluations per model ===
                                    c_index_mean  c_index_std  n_folds
model                                                                 
Random Survival Forest (tuned)            0.7045       0.0289       50
Cox PH                                    0.6859       0.0266       50
Gradient Boosted Survival Analysis        0.6855       0.0281       50

=== Paired Wilcoxon signed-rank tests (fold-matched, same splits) ===
Random Survival Forest (tuned) vs Cox PH: median diff = +0.0187, n=50, p = 6.35e-07
Gradient Boosted Survival Analysis vs Cox PH: median diff = -0.0032, n=50, p = 6.81e-01
Random Survival Forest (tuned) vs Gradient Boosted Survival Analysis: median diff = +0.0145, n=50, p = 2.99e-11
